In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from flygym_examples.spotlight_data import MotionSnippet

snippet = MotionSnippet()
sim_timestep = 1e-4

In [ ]:
import flygym
from flygym.compose import Fly, ActuatorType, FlatGroundWorld
from flygym.anatomy import Skeleton, JointPreset, ActuatedDOFPreset, AxisOrder
from flygym.compose.pose import KinematicPose
from flygym.utils.math import Rotation3D


def make_model(
    joints_preset=JointPreset.LEGS_ONLY,
    actuated_dofs_preset=ActuatedDOFPreset.LEGS_ACTIVE_ONLY,
    actuator_type=ActuatorType.POSITION,
    position_gain=50.0,
    neutral_pose_file=flygym.assets_dir / "model/pose/neutral.yaml",
    spawn_position=(0, 0, 0.7),  # xyz in mm
    spawn_rotation=Rotation3D("quat", (1, 0, 0, 0)),
):

    fly = Fly()
    axis_order = AxisOrder.YAW_PITCH_ROLL

    # Add joints
    skeleton = Skeleton(axis_order=axis_order, joint_preset=joints_preset)
    neutral_pose = KinematicPose(path=neutral_pose_file)
    fly.add_joints(skeleton, neutral_pose=neutral_pose)

    # Add position actuators and set them to the neutral pose
    actuated_dofs_list = fly.skeleton.get_actuated_dofs_from_preset(
        actuated_dofs_preset
    )
    fly.add_actuators(
        actuated_dofs_list,
        actuator_type=actuator_type,
        kp=position_gain,
        neutral_input=neutral_pose,
    )

    # Add leg adhesion
    fly.add_leg_adhesion()

    # Add visuals
    fly.colorize()
    cam = fly.add_tracking_camera()

    # Create a world and spawn the fly
    world = FlatGroundWorld()
    world.add_fly(fly, spawn_position, spawn_rotation)
    
    return fly, world, cam

In [ ]:
from flygym.warp import GPUSimulation

n_worlds = 100

fly, world, cam = make_model()
sim = GPUSimulation(world, n_worlds)

In [ ]:
renderer = sim.set_renderer(
    cam,
    playback_speed=0.2,
    output_fps=25,
    worlds=list(range(9)),  # render the first 9 worlds only
)

In [ ]:
dof_angles = snippet.get_joint_angles(
    output_timestep=sim_timestep,
    output_dof_order=fly.get_actuated_jointdofs_order(ActuatorType.POSITION),
)
# dof_angles shape: (n_total_steps, n_dofs) — smoothed, interpolated, reordered
print(f"Total steps available: {dof_angles.shape[0]}, n_dofs: {dof_angles.shape[1]}")

In [ ]:
sim_seconds = 0.1
sim_steps = int(sim_seconds / sim_timestep)


def make_target_angles_all_worlds(n_worlds, sim_steps=sim_steps):
    """Prepare the input array for all worlds.
    World 0 gets 0s-0.1s, world 1 gets 0.1s-0.2s, etc.
    """
    n_dofs = dof_angles.shape[1]
    dof_angles_all_worlds = np.zeros((n_worlds, sim_steps, n_dofs), dtype=np.float32)
    n_partitions = dof_angles.shape[0] // sim_steps
    for world_id in range(n_worlds):
        partition_idx = world_id % n_partitions
        start_idx = partition_idx * sim_steps
        end_idx = start_idx + sim_steps
        dof_angles_all_worlds[world_id, :, ...] = dof_angles[start_idx:end_idx, ...]
    return dof_angles_all_worlds


dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)

In [ ]:
from tqdm import trange  # for progress bar

fly_name = fly.name

sim.reset()

# Turn adhesion on for all 6 legs across all worlds
sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))

# As before, warm up the simulation for a few steps before applying the control inputs
sim.warmup()

# Run simulation loop
for step in trange(sim_steps):
    control_inputs = dof_angles_all_worlds[:, step, :]
    sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
    sim.step_with_profile()
    sim.render_as_needed_with_profile()

In [ ]:
sim.print_performance_report()

In [ ]:
renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

In [ ]:
def run_simulation(n_worlds):
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)
    
    fly, world, cam = make_model()
    fly_name = fly.name
    
    sim = GPUSimulation(world, n_worlds)
    renderer = sim.set_renderer(
        cam,
        playback_speed=0.2,
        output_fps=25,
        worlds=list(range(9)),  # render the first 9 worlds only
    )
    
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()
    for step in trange(sim_steps):
        control_inputs = dof_angles_all_worlds[:, step, :]
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
        sim.step_with_profile()
        sim.render_as_needed_with_profile()
    
    sim.print_performance_report()
    renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

In [ ]:
run_simulation(n_worlds=1000)

In [ ]:
def run_simulation_with_batch_rendering(n_worlds):
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds)
    
    fly, world, cam = make_model()
    fly_name = fly.name
    
    sim = GPUSimulation(world, n_worlds)
    renderer = sim.set_renderer(
        cam,
        playback_speed=0.2,
        output_fps=25,
        use_gpu_batch_rendering=True,  # enable GPU batch rendering
        worlds=None,  # None: by default, all worlds are rendered
    )
    
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()
    for step in trange(sim_steps):
        control_inputs = dof_angles_all_worlds[:, step, :]
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, control_inputs)
        sim.step_with_profile()
        sim.render_as_needed_with_profile()
    
    sim.print_performance_report()
    # still only display the first 9 worlds in the notebook
    renderer.show_in_notebook(world_id=list(range(9)), scale=0.5)

In [ ]:
run_simulation_with_batch_rendering(n_worlds=100)

In [ ]:
import warp as wp

@wp.kernel
def update_target_angles_kernel(
    dof_angles_all_worlds_gpu: wp.array3d(dtype=wp.float32),  # type: ignore
    step_counter_gpu: wp.array(dtype=wp.int32),  # type: ignore
    curr_target_angles_gpu: wp.array2d(dtype=wp.float32),  # type: ignore
):
    world_id, actuator_id = wp.tid()
    step = step_counter_gpu[0]
    my_target = dof_angles_all_worlds_gpu[world_id, step, actuator_id]
    curr_target_angles_gpu[world_id, actuator_id] = my_target

@wp.kernel
def increment_counter_kernel(
    step_counter_gpu: wp.array(dtype=wp.int32),  # type: ignore
):
    step_counter_gpu[0] = step_counter_gpu[0] + 1

In [ ]:
from time import perf_counter_ns


def run_simulation_fullgpu(n_worlds, enable_rendering):
    sim_steps = 300  # for timing, run shorter sim to ensure it fits in GPU memory
    dof_angles_all_worlds = make_target_angles_all_worlds(n_worlds, sim_steps)
    n_dofs = dof_angles_all_worlds.shape[-1]

    fly, world, cam = make_model()
    fly_name = fly.name
    sim = GPUSimulation(world, n_worlds)

    if enable_rendering:
        renderer = sim.set_renderer(
            cam,
            playback_speed=0.2,
            output_fps=25,
            use_gpu_batch_rendering=True,
        )

    # Turn adhesion on for all 6 legs across all worlds
    sim.set_leg_adhesion_states(fly_name, np.ones((n_worlds, 6), dtype=np.float32))
    sim.warmup()

    dof_angles_all_worlds_gpu = wp.array(dof_angles_all_worlds)
    curr_target_angles_gpu = wp.zeros((n_worlds, n_dofs), dtype=wp.float32)
    step_counter = wp.array([0], dtype=wp.int32)

    with wp.ScopedCapture() as advance_sim_capture:
        wp.launch(
            update_target_angles_kernel,
            dim=(n_worlds, n_dofs),
            inputs=[dof_angles_all_worlds_gpu, step_counter],
            outputs=[curr_target_angles_gpu],
        )
        sim.set_actuator_inputs(fly_name, ActuatorType.POSITION, curr_target_angles_gpu)
        sim.step()
        wp.launch(increment_counter_kernel, dim=1, outputs=[step_counter])

    start_time = perf_counter_ns()
    for step in trange(sim_steps):
        wp.capture_launch(advance_sim_capture.graph)
        if enable_rendering:
            sim.render_as_needed()
    end_time = perf_counter_ns()
    walltime_seconds = (end_time - start_time) / 1e9
    overall_throughput = n_worlds * sim_steps / walltime_seconds  # steps per sec
    overall_realtime_factor = overall_throughput * sim_timestep  # x realtime

    print(
        f"Simulated {sim_steps} steps * {n_worlds} worlds in {walltime_seconds:.2f}s\n"
        f"Overall throughput: {overall_throughput:.2f} steps/s "
        f"({overall_realtime_factor:.2f}x realtime)"
    )

In [ ]:
run_simulation_fullgpu(n_worlds=1000, enable_rendering=True)

In [ ]:
run_simulation_fullgpu(n_worlds=1000, enable_rendering=False)